In [3]:
#PostgreSQL Tables extraction
from database import extract_table
import pandas as pd

customers = extract_table("customers")
bookings = extract_table("bookings")
payments = extract_table("payments")
feedback = extract_table("feedback")
services = extract_table("services")

customers.head()

# print(feedback.head())


,customer_id,name,phone,email,created_at
0,1,Allison Hill,0830433218,allisonhill@gmail.com,2025-10-30 14:15:20
1,2,Noah Rhodes,0839600133,noahrhodes@gmail.com,2025-07-15 05:29:03
2,3,Angie Henderson,0828386379,angiehenderson@gmail.com,2025-08-09 08:20:30
3,4,Daniel Wagner,0740265423,danielwagner@gmail.com,2026-02-04 13:57:50
4,5,Cristian Santos,0761161559,cristiansantos@gmail.com,2025-09-09 05:06:34


In [4]:
#store raw data as csv files 
#create folder path
import os

os.makedirs("../data/raw", exist_ok=True)
customers.to_csv("../data/raw/customers_raw.csv",index=False)
payments.to_csv("../data/raw/payments_raw.csv",index=False)
feedback.to_csv("../data/raw/feedback_raw.csv",index=False)
bookings.to_csv("../data/raw/bookings_raw.csv",index=False)

In [5]:
#DATA CLEANING

#Under bookings, we don't need the "created_at" column when we have "scheduled_at", 
#since by default the column was assigned to CURRENT_TIMESTAMP

#we collapse the column
bookings=bookings.drop(columns='created_at')
bookings.head()

,booking_id,customer_id,service_id,scheduled_date,status
0,1,39,3,2026-02-01 06:26:51,COMPLETED
1,2,29,3,2026-01-23 21:18:24,CANCELLED
2,3,15,2,2026-02-17 03:53:32,CANCELLED
3,4,43,1,2026-02-14 05:40:00,CANCELLED
4,5,8,1,2026-02-12 03:08:46,COMPLETED


In [6]:
#Dropping duplicates
customers=customers.drop_duplicates(subset="customer_id")
bookings=bookings.drop_duplicates(subset="booking_id")
payments=payments.drop_duplicates(subset="payment_id")


In [7]:
# Normalize email format
customers["email"] = customers["email"].str.lower().str.strip()
# Detecting missing values
customers.isnull().sum()
services.isnull().sum()
bookings.isnull().sum()
payments.isnull().sum()
feedback.isnull().sum()
# Ensuring rating is numeric
# if a value can't be converted to a number, turn it into NaN
feedback['rating']=pd.to_numeric(feedback['rating'],errors='coerce')
feedback.isna().sum()
#remove invalid ratings
feedback=feedback[(feedback['rating']>=1)& (feedback['rating']<=5)]


In [8]:
#Transform the data (merge)
def transform(customers,bookings,services,payments,feedback):
    df=bookings.merge(customers, on='customer_id',how='left')
    df=df.merge(payments,on='booking_id',how='left')
    df=df.merge(feedback,on='booking_id',how='left')
    
    
    return df
main_df=transform(customers, bookings, services,payments, feedback )
main_df


,booking_id,customer_id,service_id,scheduled_date,status,name,phone,email,created_at,payment_id,amount,payment_method,payment_date,feedback_id,rating
0,1,39,3,2026-02-01 06:26:51,COMPLETED,Elizabeth Fowler,0747510799,elizabethfowler@gmail.com,2026-02-18 12:34:39,1.0,300.0,Transfer,2026-02-02 06:48:57,1.0,3.0
1,2,29,3,2026-01-23 21:18:24,CANCELLED,Zachary Hicks,0761333872,zacharyhicks@gmail.com,2025-03-19 17:23:29,NaN,NaN,NaN,NaT,NaN,NaN
2,3,15,2,2026-02-17 03:53:32,CANCELLED,Dylan Miller,0722691669,dylanmiller@gmail.com,2025-05-20 13:46:40,NaN,NaN,NaN,NaT,NaN,NaN
3,4,43,1,2026-02-14 05:40:00,CANCELLED,Sherry Decker,0724493534,sherrydecker@gmail.com,2025-08-22 18:33:58,NaN,NaN,NaN,NaT,NaN,NaN
4,5,8,1,2026-02-12 03:08:46,COMPLETED,Gina Moore,0741316475,ginamoore@gmail.com,2025-05-27 11:36:03,2.0,150.0,Cash,2026-02-13 03:30:52,2.0,5.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,37,3,2026-02-12 19:14:01,CANCELLED,Sherri Baker,0739909169,sherribaker@gmail.com,2025-04-12 14:11:19,NaN,NaN,NaN,NaT,NaN,NaN
196,197,33,1,2026-03-19 14:11:32,COMPLETED,Tricia Valencia,0784746872,triciavalencia@gmail.com,2025-11-07 00:16:40,67.0,150.0,Cash,2026-03-20 14:33:38,67.0,1.0
197,198,42,1,2026-02-11 20:10:16,CANCELLED,Fred Smith,0828412411,fredsmith@gmail.com,2025-10-01 23:38:05,NaN,NaN,NaN,NaT,NaN,NaN
198,199,44,1,2026-01-22 03:47:00,PENDING,Anthony Humphrey,0794016400,anthonyhumphrey@gmail.com,2025-05-25 13:51:05,NaN,NaN,NaN,NaT,NaN,NaN


In [9]:
#collapse unnecessary columns
main_df=main_df.drop(columns='created_at')

main_df=main_df.drop(columns='feedback_id')
#reoder columns
desired_order=['booking_id','customer_id', 'service_id','name','phone','email','status','payment_id',
             'amount','payment_method','rating','scheduled_date','payment_date']
main_df=main_df[desired_order]
main_df





,booking_id,customer_id,service_id,name,phone,email,status,payment_id,amount,payment_method,rating,scheduled_date,payment_date
0,1,39,3,Elizabeth Fowler,0747510799,elizabethfowler@gmail.com,COMPLETED,1.0,300.0,Transfer,3.0,2026-02-01 06:26:51,2026-02-02 06:48:57
1,2,29,3,Zachary Hicks,0761333872,zacharyhicks@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-01-23 21:18:24,NaT
2,3,15,2,Dylan Miller,0722691669,dylanmiller@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-02-17 03:53:32,NaT
3,4,43,1,Sherry Decker,0724493534,sherrydecker@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-02-14 05:40:00,NaT
4,5,8,1,Gina Moore,0741316475,ginamoore@gmail.com,COMPLETED,2.0,150.0,Cash,5.0,2026-02-12 03:08:46,2026-02-13 03:30:52
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,37,3,Sherri Baker,0739909169,sherribaker@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-02-12 19:14:01,NaT
196,197,33,1,Tricia Valencia,0784746872,triciavalencia@gmail.com,COMPLETED,67.0,150.0,Cash,1.0,2026-03-19 14:11:32,2026-03-20 14:33:38
197,198,42,1,Fred Smith,0828412411,fredsmith@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-02-11 20:10:16,NaT
198,199,44,1,Anthony Humphrey,0794016400,anthonyhumphrey@gmail.com,PENDING,NaN,NaN,NaN,NaN,2026-01-22 03:47:00,NaT


In [10]:
#column renaming
main_df=main_df.rename(columns={
    'name':'customer_name',
    'status':'payment_status',
    'amount':'payment_amount',
    'rating':'customer_rating'
    
})

In [19]:
#date time formatting
import pandas as pd
main_df['payment_date']=pd.to_datetime(main_df['payment_date'], errors='coerce')
main_df['scheduled_date']=pd.to_datetime(main_df['scheduled_date'],errors='coerce')
main_df

,booking_id,customer_id,service_id,customer_name,phone,email,payment_status,payment_id,payment_amount,payment_method,customer_rating,scheduled_date,payment_date
0,1,39,3,Elizabeth Fowler,0747510799,elizabethfowler@gmail.com,COMPLETED,1.0,300.0,Transfer,3.0,2026-02-01 06:26:51,2026-02-02 06:48:57
1,2,29,3,Zachary Hicks,0761333872,zacharyhicks@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-01-23 21:18:24,NaT
2,3,15,2,Dylan Miller,0722691669,dylanmiller@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-02-17 03:53:32,NaT
3,4,43,1,Sherry Decker,0724493534,sherrydecker@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-02-14 05:40:00,NaT
4,5,8,1,Gina Moore,0741316475,ginamoore@gmail.com,COMPLETED,2.0,150.0,Cash,5.0,2026-02-12 03:08:46,2026-02-13 03:30:52
...,...,...,...,...,...,...,...,...,...,...,...,...,...
195,196,37,3,Sherri Baker,0739909169,sherribaker@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-02-12 19:14:01,NaT
196,197,33,1,Tricia Valencia,0784746872,triciavalencia@gmail.com,COMPLETED,67.0,150.0,Cash,1.0,2026-03-19 14:11:32,2026-03-20 14:33:38
197,198,42,1,Fred Smith,0828412411,fredsmith@gmail.com,CANCELLED,NaN,NaN,NaN,NaN,2026-02-11 20:10:16,NaT
198,199,44,1,Anthony Humphrey,0794016400,anthonyhumphrey@gmail.com,PENDING,NaN,NaN,NaN,NaN,2026-01-22 03:47:00,NaT


In [20]:
os.makedirs("../data/processed", exist_ok=True)
customers.to_csv("../data/processed/customers.csv",index=False)
main_df.to_csv("../data/processed/final_data.csv",index=False)